# Indiana Pines

1. Import dependencies

In [ ]:
import random
import torch
import multiprocessing
import time
import numpy as np

import torch.utils.data as data

from src.util.torch import resolve_torch_device
from src.util.hsi import (
    extract_patches,
    reduce_hsi_dim,
    preprocess_hsi,
    PreProcessType,
    DimReductionType,
)
from src.data.indian_pines import load_indian_pines
from src.visualization.plot import (
    plot_epoch_generic,
)

from src.util.reporting import (
    create_model_name,
    report_run,
    read_report_to_show,
)
from src.util.list_ext import smooth_moving_average
from src.util.torch import save_model
from src.trainer.base_trainer import AdamOptimizedModule
from src.data.dataset_decorator import RandomFlipDatasetDecorator, HsiMaeAdapterDatasetDecorator
from src.trainer.base_trainer import LrSchedulerProvider
from src.model.hsimae import HSIMAE
from src.trainer.hsimae_trainer import HsiMaeTrainer

2. Prepare env

In [ ]:
run_desk = "Double check 0.5 mask_ratio"

In [ ]:
learning_rate = 5e-3
weight_decay = 0
num_epochs = 120

In [ ]:
encoder_dim = 256
encoder_depth = 12
decoder_dim = 64
decoder_depth = 8
s_depth = 9
mask_ratio = 0.5
b_patch_size = 5

In [ ]:
batch_size = 512
patch_size = 9
target_dim = 75

examples_per_class = {}

pre_process_type = PreProcessType.STANDARTIZATION
dim_reduction_type = DimReductionType.PCA

In [ ]:
random_seed = 42

random.seed(random_seed)
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = resolve_torch_device()

In [ ]:
epoch_seconds = int(time.time())

In [ ]:
torch.cuda.empty_cache()

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
f"Device is {device}"

3. Load dataset

In [ ]:
image, labels = load_indian_pines()

image_h, image_w, image_c = image.shape

In [ ]:
_, image = preprocess_hsi(image, pre_process_type)

In [ ]:
_, target_dim, image = reduce_hsi_dim(
    image, target_dim, dim_reduction_type, random_seed
)

In [ ]:
x, y = extract_patches(image, labels, patch_size=patch_size)

In [ ]:
num_classes = len(np.unique(y))

f"Number of classes {num_classes}"

In [ ]:
x_tensor = torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2) 
y_tensor = torch.tensor(y, dtype=torch.long)

In [ ]:
cpu_count = multiprocessing.cpu_count()

f"Setting num_workers to {cpu_count}"

In [ ]:
full_dataset = data.TensorDataset(x_tensor, y_tensor)

full_loader = data.DataLoader(
    HsiMaeAdapterDatasetDecorator(RandomFlipDatasetDecorator(full_dataset)),
    batch_size=batch_size,
    shuffle=False,
    num_workers=cpu_count,
    persistent_workers=True,
)

4. Train model

In [ ]:
backbone = HSIMAE(
    img_size=patch_size,
    patch_size=3,
    in_chans=1,
    bands=target_dim,
    b_patch_size=b_patch_size,
    embed_dim=encoder_dim,
    depth=encoder_depth,
    num_heads=encoder_dim // 16,
    s_depth=s_depth,
    decoder_embed_dim=decoder_dim,
    decoder_depth=decoder_depth,
    decoder_num_heads=decoder_dim // 8,
    norm_pix_loss=True,
    trunc_init=True,
)

In [ ]:
model = AdamOptimizedModule(
    net=backbone,
    lr=learning_rate,
    weight_decay=weight_decay,
    scheduler=LrSchedulerProvider(
        t_initial=num_epochs,
        lr_min=1e-6,
        warmup_t=int(np.ceil(0.1 * num_epochs)),
        warmup_lr_init=learning_rate * 0.01,
    ),
).to(device)

In [ ]:
trainer = HsiMaeTrainer(
    epochs=num_epochs,
    mask_ratio=mask_ratio,
    device=device,
)

In [ ]:
feedback = trainer.fit(model, full_loader)

In [ ]:
smothed_train = smooth_moving_average([it.train["train_loss"] for it in feedback.history], 1)

plot_epoch_generic(smothed_train, desc="Loss")

In [ ]:
validation_result = trainer.validate(model, full_loader)

validation_result

5. Save weights

In [ ]:
saved_model_name = f"indian_pines_hsimae_encoder_{run_desk.replace(" ", "_").lower()}"

save_model(backbone, saved_model_name)

6. Write report

In [ ]:
model_name = create_model_name("indian_pines_mae", {})

model_category = "hsimae"

In [ ]:
run_params = {
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "patch_size": patch_size,
    "target_dim": target_dim,
    "mask_ratio": mask_ratio,
    "pre_process_type": str(pre_process_type),
    "dim_reduction_type": str(dim_reduction_type),
    "saved_model_name": saved_model_name,
}

run_params = run_params | model.get_params()

In [ ]:
report_run(
    model_name=model_name,
    model_category=model_category,
    run_desc=run_desk,
    run_params=run_params,
    run_metrics={"loss": validation_result["eval_loss"]},
)

In [ ]:
read_report_to_show(model_name, sort_by_metric="loss", ascending=True)

In [ ]:
read_report_to_show(
    model_name, sort_by_metric="loss", model_category=model_category, ascending=True
)